These notes begin with ordinary LLM requests and build up to tool calling. Using the OpenAI Python client with [DeepSeek's API](https://www.subconscious.dev/providers/deepseek), we chain messages, reuse prompt templates, and let the model request a word-counting function. The example distinguishes the model's request from the application code that executes it and returns the result.

We then examine how tool calls fit into a goal-directed feedback loop, and how orchestration patterns organize that work. The remaining sections consider how to trace a run, evaluate its outcome, and enforce limits on execution. These distinctions help us assess when an agent adds something beyond a single LLM call or a fixed workflow.


## The OpenAI client

The OpenAI client requires an API key and the base url of the model provider:

In [1]:
from openai import AsyncOpenAI
from atlas.vault import get_secret

MODEL_NAME = "deepseek-flash"
BASE_URL = "https://api.deepseek.com"
API_KEY = get_secret("DEEPSEEK_API_KEY")
client = AsyncOpenAI(api_key=API_KEY, base_url=BASE_URL)

### Our first LLM call

Recall that behind the API is a language model that performs text completion. Here we choose a specific model under `model` then the **message chain** is passed to that model as prompt. The first message[^llm_focus] is usually the **system prompt** with role `system` containing things like persona, instructions, behavioral constraints, etc. Then, we can append our messages which has role `user`. 

Testing out a bizzare system prompt:

[^llm_focus]: LLMs [focus highly](/posts/ai-agents-foundations.html#fig-lost-in-the-middle) on the first and last messages of the message history. This is good since its natural for us to construct the message history with instructions at the very start of the conversation, and the most recent user query at the last.

In [2]:
response = await client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a loyal dog."}, 
        {"role": "user", "content": "Hey, boy!"}
    ],
)

print(response.choices[0].message.content)

*ears perk up, tail wagging* Woof! Hey there! I’m right here, buddy! 🐾


The client allows further configs such as whether we **stream response** or not. Also whether the model should use **reasoning** along with a guidance on how much reasoning effort the model should use in its response (generally, the higher the reasoning, the better). The output is usually markdown compatible, which we render here.

In [3]:
from IPython.display import display, Markdown

response = await client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "Tell me a fun fact."},
    ],
    stream=False,
    reasoning_effort="low",
    extra_body={"thinking": {"type": "enabled"}},
)

display(Markdown("> " + response.choices[0].message.content))

> Fun fact: **A day on Venus is longer than a year on Venus.** Venus takes about **243 Earth days** to rotate once on its axis, but only about **225 Earth days** to orbit the Sun. So if you lived on Venus, you’d celebrate your birthday before you finished a single “day.”

For testing we can set `n > 0` in `client.chat.completions.create` to get a sample of `n` responses for the same prompt. But by default `n=1`. Hence, `response.choices` is a singleton and `[0]` just gets the sole response. Here we also see the role of the model is `assistant`. Moreover, DeepSeek exposes its **reasoning trace** in `reasoning_content`:

In [8]:
from rich import print_json
print_json(response.choices[0].model_dump_json())

{
  "finish_reason": "stop",
  "index": 0,
  "logprobs": null,
  "message": {
    "content": "Octopuses have three hearts: two pump blood to their gills, and one pumps blood to the rest of the body. The main heart actually stops beating when the octopus swims—which is part of why they often prefer crawling along the seafloor.",
    "refusal": null,
    "role": "assistant",
    "annotations": null,
    "audio": null,
    "function_call": null,
    "tool_calls": null,
    "reasoning_content": "We need answer fun fact concise. Need comply. Pick fun fact. Maybe \"Octopuses have three hearts; two pump blood to gills, one to rest; the main one stops when swimming, which is why they prefer crawling.\" That's fun. Need answer."
  }
}

The following examples repeat the same request and display steps, so let's put them in a function. `chat` accepts the message history and returns the full response for chaining. Its keyword arguments select the model and its reasoning settings. With `verbose=False`, we can receive the response without printing it.


In [5]:
async def chat(
    messages,
    *,
    model=MODEL_NAME,
    reasoning_effort="low",
    thinking_type="enabled",
    verbose=True
):
    response = await client.chat.completions.create(
        model=model,
        messages=messages,
        stream=False,
        reasoning_effort=reasoning_effort,
        extra_body={"thinking": {"type": thinking_type}},
    )

    # expose reasoning trace and response content as markdown
    if verbose:
        message = response.choices[0].message
        if thinking_type == "enabled":
            reasoning = message.model_dump().get("reasoning_content").replace("\n", " ")
            print("Reasoning:")
            display(Markdown("> " + reasoning))

        content = message.content or ""
        content = content.replace("\n", " ")
        print("Response:")
        display(Markdown("> " + content))
        
    return response

### Chaining messages

The sum of all turns is a conversation, or **context** (also called a "chat" with the agent UI) (@fig-week-1-introduction-to-ai-agents-edefadede3ef490c9c36e905666d6af4). Chatting with the model is possible in spite of it being stateless since we're feeding the entire conversation (all prompts and model and user responses) at each turn. 

![The message chain is sent to the model with each new prompt. **Source:** [*Flet agent got eyes and ears*][flet-context-source] by Feodor Fitsner.](../images/post/week-1-introduction-to-ai-agents-edefadede3ef490c9c36e905666d6af4.png){#fig-week-1-introduction-to-ai-agents-edefadede3ef490c9c36e905666d6af4 fig-alt="The context window contains turns 1, 2, and 3 followed by the new prompt. The full sequence is sent to the model on each call."}

Let's make the same message as above:

[flet-context-source]: https://flet.dev/blog/flet-agent-got-eyes-and-ears/


In [6]:
SYSTEM_PROMPT = "You are a helpful assistant"
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Tell me a fun fact."},
]

# First call
response = await chat(messages)

Reasoning:


> We need answer fun fact concise. Need comply. Pick fun fact. Maybe "Octopuses have three hearts; two pump blood to gills, one to rest; the main one stops when swimming, which is why they prefer crawling." That's fun. Need answer.

Response:


> Octopuses have three hearts: two pump blood to their gills, and one pumps blood to the rest of the body. The main heart actually stops beating when the octopus swims—which is part of why they often prefer crawling along the seafloor.

Let's create a user response and chain it to the message history:

In [7]:
follow_up = "That's cool!"

msg = response.choices[0].message
messages.append({"role": msg.role, "content": msg.content})
messages.append({"role": "user", "content": follow_up})

# Second call
await chat(messages);

Reasoning:


> We need answer to user. They said "That's cool!" Need respond friendly, maybe add another fun fact or acknowledge. Desired oververbosity 5 but simple. Should not be too long. Could say glad you liked it, offer related fact: octopus blood blue due copper hemocyanin. Maybe ask if want more. Keep concise.

Response:


> Glad you liked it! Here's a related one: octopus blood is blue because it uses copper-based hemocyanin instead of iron-based hemoglobin to carry oxygen. Want another fun fact?

The response of the model is coherent since it has the full message history.

### Prompt templates

Templates let us write reusable instructions while giving our code control over how each prompt takes shape. We can fill placeholders with new inputs, use conditionals to include relevant context, and loop over examples or documents. The same template can then produce prompts tailored to different requests. Python f-strings handle simple substitutions, while **template engines** such as [Jinja2](https://realpython.com/primer-on-jinja-templating/#render-your-first-jinja-template) let us express conditionals and loops directly within the template.

In [9]:
async def explain(topic: str, audience: str, max_words: int = 100):
    SYSTEM_PROMPT = "You are a helpful assistant"
    prompt = (
        f"Explain {topic} to {audience}. "
        "Include one realistic concrete example relevant to the audience. "
        f"Keep the answer under {max_words} words."
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    print("Prompt:")
    display(Markdown("> " + prompt))
    return await chat(messages, reasoning_effort="high")


await explain(topic="AI agents", audience="a Python programmer");

Prompt:


> Explain AI agents to a Python programmer. Include one realistic concrete example relevant to the audience. Keep the answer under 100 words.

Reasoning:


> We need answer under 100 words. Explain AI agents to Python programmer, concrete example. Need concise. Count words. We need mention agent loop LLM + tools + memory + planning, example maybe Python agent using function-calling to query DB and write report. Under 100 words. Let's craft ~70 words. Ensure realistic. "An AI agent is a Python loop around an LLM: give it tools (functions), memory/state, and a goal; it decides which tool to call, observes results, repeats until done. Example: a customer-support agent using OpenAI function calling. Tools: get_order(id), issue_refund(id). User asks 'Refund order 123.' Agent calls get_order(123), checks status, calls issue_refund(123), returns confirmation. You implement tools as Python functions; the LLM selects and sequences them." Count. Under 100. Good. Need final only.

Response:


> An AI agent is a Python loop around an LLM: give it tools (functions), memory/state, and a goal; it decides which tool to call, observes results, and repeats until done.  Example: a support agent using function calling. Tools: `get_order(id)` and `issue_refund(id)`. User says “Refund order 123.” The agent calls `get_order(123)`, checks eligibility, calls `issue_refund(123)`, then confirms. You implement the tools as normal Python functions; the LLM selects and sequences them.

Pretty good. Trying again with a different `audience`:

In [10]:
await explain(
    topic="AI agents",
    audience="a research mathematician. Use plain Unicode notation (π and →) and no LaTeX",
);

Prompt:


> Explain AI agents to a research mathematician. Use plain Unicode notation (π and →) and no LaTeX. Include one realistic concrete example relevant to the audience. Keep the answer under 100 words.

Reasoning:


> We need answer under 100 words, plain Unicode, no LaTeX, explain AI agents to research mathematician, include realistic concrete example relevant. Need concise. Need define agent: LLM + tools + loop, policy mapping observations to actions, goal-directed, can call compute, search, code, etc. Example: mathematician asks agent to conjecture/verify identity: agent searches literature, runs symbolic/numeric checks in Python, returns proof sketch/counterexample, repeats. Keep <100 words. Count. Need plain unicode π and →. No LaTeX. Let's craft ~70 words.  Need "AI agent" explanation: a control loop around a language model: given goal, model chooses actions (search, code, query databases), observes results, updates plan, repeats until stopping criterion. Policy π: state→action. Example relevant: agent given "test whether ∑..."? Realistic concrete. Maybe "prove/disprove ∑_{n≤N} ..."? Avoid LaTeX. Use plain text. "Check whether π is irrational"—hard. Better: "Given a conjectured identity involving π, an agent searches OEIS/arXiv, writes Python/SymPy checks for many parameter values, and if a discrepancy appears, returns a counterexample; otherwise summarizes evidence and next proof steps." That's realistic. Under 100 words. Count words. Let's write.  Need include π and →. "An AI agent is an LLM inside a control loop: it maps observations to actions (π: state → action), uses tools (search, code, databases), and iterates toward a goal until a stopping rule." Example.  Word count: An(1) AI2 agent3 is4 an5 LLM6 inside7 a8 control9 loop:10 it11 maps12 observations13 to14 actions15 (π:16 state17 →18 action),19 uses20 tools21 (search,22 code,23 databases),24 and25 iterates26 toward27 a28 goal29 until30 a31 stopping32 rule.33 Example:34 given35 a36 conjectured37 identity38 involving39 π,40 an41 agent42 searches43 OEIS/arXiv,44 writes45 SymPy46 checks47 across48 parameter49 values,50 and51 returns52 either53 a54 counterexample55 or56 a57 evidence58 summary59 plus60 next61 proof62 steps.63 That's 63 words. Good. Need maybe "plain Unicode notation" yes. No LaTeX. Good. Final.

Response:


> An AI agent is an LLM inside a control loop: it maps observations to actions (π: state → action), uses tools (search, code, databases), and iterates toward a goal until a stopping rule. Example: given a conjectured identity involving π, an agent searches OEIS/arXiv, writes SymPy checks across parameter values, and returns either a counterexample or an evidence summary plus next proof steps.

That's pretty cool.

:::{.callout-note}
Interestingly, the reasoning model is actually counting. Counting like this using reasoning tokens seems wasteful, esp. with high word count. The alternative is the model doing a **tool call** to bash / Python to write a script for counting the words. This is discussed further below.
:::

## What is an AI agent?

So far, `chat` and `explain` produce answers when we call them. We decide which prompt comes next, and when the conversation ends. In contrast, an **AI agent** chooses some of those next steps itself: it works toward a **goal**, uses the **tools** available to it, and responds to what it **observes**. To achieve its goal, the model can select an action based on the current state, while the application controls which actions are available and when execution must stop[^subagents]. Therefore, a useful working description is:

> An agent is an LLM with tools in a feedback loop to achieve a goal.

[^subagents]: In advanced settings, an **orchestrator agent** can trigger workflows consisting of **specialist subagents** that process a request. Also see [this](/posts/week-1-introduction-to-ai-agents.html#workflow-patterns).

### Tool calling

Earlier, the model tried to keep an explanation under a word limit by counting in its reasoning. We can give it a Python function instead. A **tool** is an operation the application makes available to the model: it might execute a function, search documents, query a database, or call an API. But since the controller is a text generation model, it doesn't really execute the function. Instead, given a list of available tools (or tool schema), it responds with the *name* of the function and the *arguments* that it would call that function with. Hence, everything is text-based (@fig-week-1-introduction-to-ai-agents-0e25d535d05b499d87763ecc9ebda2cc).

![**A brain-in-a-vat analogy for tool calling.** The LLM *ideates* a function call—it generates the function name and arguments rather than executing anything. The application validates and runs the call, then returns its result as input to the model. **Source:** [*Brain in a vat*][brain-in-vat-source], original by Alexander Wivel, vector by Pduive23, [CC BY 3.0][brain-in-vat-license]; unchanged.](../images/post/week-1-introduction-to-ai-agents-0e25d535d05b499d87763ecc9ebda2cc.svg){#fig-week-1-introduction-to-ai-agents-0e25d535d05b499d87763ecc9ebda2cc fig-alt="A brain suspended in a vat, connected by wires to an external computer." width="400"}

[brain-in-vat-source]: https://commons.wikimedia.org/wiki/File:Braininvat.svg
[brain-in-vat-license]: https://creativecommons.org/licenses/by/3.0/


#### Tool definition

This is a name, a description, and a JSON Schema describing the arguments. OpenAI's [function-calling guide](https://developers.openai.com/api/docs/guides/function-calling)[^deepseek] is the primary reference for tool definitions and the call/result exchange. Since we're using Chat Completions, the function definition is nested under `tools[].function` (the Responses API uses a different envelope). Here, `count_words` counts whitespace-separated chunks with `text.split()` for simplicity.

[^deepseek]: Other OpenAI-compatible providers such as DeepSeek follow the same schema.

In [11]:
def count_words(text: str) -> int:
    return len(text.split())

# See function calling guide for the OpenAI API
tools = [
    {
        "type": "function",
        "function": {
            "name": "count_words",
            "description": "Count whitespace-separated words in the supplied text.",
            "parameters": {
                "type": "object",
                "properties": {"text": {"type": "string"}},
                "required": ["text"],
                "additionalProperties": False,
            },
        },
    }
]

tools_registry = {
    "count_words": count_words
}

::: {.callout-note #what-a-tool-schema-can-guarantee title="What a tool schema can guarantee"}
A tool's `parameters` describe its arguments using **JSON Schema**: types, required fields, and allowed values. Clear names and descriptions guide tool choice; provider support determines which constraints are enforced during generation. See OpenAI's [function-definition guide](https://developers.openai.com/api/docs/guides/function-calling#defining-functions).

**A valid shape does not guarantee a correct or authorized action.** Both `A123` and `B456` satisfy a string field, even when only one is relevant or accessible to the user. The application still validates arguments, enforces permissions, and checks results against the goal. Focused context helps the model choose; a schema or stronger model cannot replace those checks.
:::


Next, we extend `chat` to support tool use:

In [12]:
async def chat(
    messages,
    *,
    model=MODEL_NAME,
    reasoning_effort="low",
    thinking_type="enabled",
    verbose=True,
    tools=None,
    tool_choice="auto"
):
    response = await client.chat.completions.create(
        model=model,
        messages=messages,
        stream=False,
        reasoning_effort=reasoning_effort,
        tools=tools,
        tool_choice=tool_choice,
        extra_body={"thinking": {"type": thinking_type}},
    )

    # expose reasoning trace and response content as markdown
    if verbose:
        message = response.choices[0].message
        reasoning = message.model_dump().get("reasoning_content")
        if thinking_type == "enabled" and reasoning:
            print("Reasoning:")
            display(Markdown("> " + reasoning.replace("\n", " ")))

        # Display the requested tool calls or the final answer.
        print("Response:")
        if response.choices[0].finish_reason == "tool_calls":
            for tool_call in message.tool_calls:
                print(f"Tool call ID: {tool_call.id}")
                print(tool_call.function.model_dump_json(indent=2))
        else:
            content = message.content or ""
            content = content.replace("\n", " ")
            display(Markdown("> " + content))
        
    return response

#### Tool call exchange

1. Send the conversation and tool definitions to the model.
2. Receive an assistant tool call message with function name and JSON-encoded arguments.
3. Validate the request, run the Python function, and append its result as a `tool` message[^tool_call_id].
4. Send the updated history back so the model can answer using the result.


[^tool_call_id]: The `tool_call_id` links that result to the original request.

![An AI agent’s tool-calling loop. **Source:** [*Flet agent got eyes and ears*][flet-agent-loop-source] by Feodor Fitsner.](../images/post/week-1-introduction-to-ai-agents-8356420be60f4153bd46ea02abfa4651.png){#fig-week-1-introduction-to-ai-agents-8356420be60f4153bd46ea02abfa4651 fig-alt="The user sends a prompt to the agent, which adds the system prompt and available tools before calling the model. When the model requests a tool, the agent runs it and passes the result back. This repeats until the model returns a final answer, which the agent sends to the user."}

[flet-agent-loop-source]: https://flet.dev/blog/flet-agent-got-eyes-and-ears/


In [13]:
import json

example = "An agent uses tools and feedback to complete tasks."
messages = [
    {"role": "system", "content": "Use count_words, then report the word count."},
    {"role": "user", "content": f"How many words are in this text?\n\n{example}"},
]

response = await chat(messages, tools=tools)
messages.append(response.choices[0].message.model_dump())

Reasoning:


> We need count words. Use tool. Text: "An agent uses tools and feedback to complete tasks." Count whitespace-separated words = An1 agent2 uses3 tools4 and5 feedback6 to7 complete8 tasks9. 9. But must use tool.

Response:
Tool call ID: call_00_gbOgtTeLr2ZM7pfuTuP80854
{
  "arguments": "{\"text\": \"An agent uses tools and feedback to complete tasks.\"}",
  "name": "count_words"
}


**Execution.** We wrap the result with role `tool` which we send back to the LLM.

In [14]:
tool_call = response.choices[0].message.tool_calls[0]
fn_schema = tool_call.function
fn = tools_registry[fn_schema.name]
result = fn(**json.loads(fn_schema.arguments))

messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps({"result": result}),
})

print_json(json.dumps(messages[-1]))

{
  "role": "tool",
  "tool_call_id": "call_00_gbOgtTeLr2ZM7pfuTuP80854",
  "content": "{\"result\": 9}"
}

:::{.callout-warning}
Ideally, we also **validate** the `tool_call` (e.g. arguments are proper and the function exists) in this step. But here we're being lazy (>ᴗ•). Also, possibly retrying if the execution fails.
:::

**Synthesis.** The model responds based on the tool call and the result (matched by id) and the surrounding context:

In [15]:
response = await chat(messages, thinking_type="disabled")

Response:


> The text contains **9 words**.

These are the messages we sent to the model for synthesis.

In [16]:
print_json(json.dumps(messages))

[
  {
    "role": "system",
    "content": "Use count_words, then report the word count."
  },
  {
    "role": "user",
    "content": "How many words are in this text?\n\nAn agent uses tools and feedback to complete tasks."
  },
  {
    "content": "",
    "refusal": null,
    "role": "assistant",
    "annotations": null,
    "audio": null,
    "function_call": null,
    "tool_calls": [
      {
        "id": "call_00_gbOgtTeLr2ZM7pfuTuP80854",
        "function": {
          "arguments": "{\"text\": \"An agent uses tools and feedback to complete tasks.\"}",
          "name": "count_words"
        },
        "type": "function",
        "index": 0
      }
    ],
    "reasoning_content": "We need count words. Use tool. Text: \"An agent uses tools and feedback to complete tasks.\" Count whitespace-separated words = An1 agent2 uses3 tools4 and5 feedback6 to7 complete8 tasks9. 9. But must use tool."
  },
  {
    "role": "tool",
    "tool_call_id": "call_00_gbOgtTeLr2ZM7pfuTuP80854",
    "content": "{\"result\": 9}"
  }
]

::: {.callout-tip #async-and-parallel-tool-calls title="Async and parallel tool calls"}
An `await` expression suspends the current coroutine until a result is available; other tasks may run in the meantime. Two calls awaited one after the other still run in sequence. Suppose `lookup_order` is asynchronous. We may await two independent lookups together:

```python
import asyncio

first, second = await asyncio.gather(
    lookup_order("A123"),
    lookup_order("B456"),
)
```

Neither lookup uses the result of the other, so their waiting times can overlap.

In OpenAI Chat Completions, [`parallel_tool_calls=True`](https://developers.openai.com/api/docs/guides/function-calling#parallel-function-calling) permits several calls in one assistant message. The `tool_calls` array supplies their names and arguments, but no dependency information. The harness chooses how to execute them using [application-defined rules]{.mark} about the tools and the resources they affect. Such rules might allow independent order lookups to overlap, but serialize updates to the same order. Without this information, the array alone is insufficient to determine an execution order.

If a second call needs a tracking number returned by the first, merely executing the batch sequentially does not supply it: both calls' arguments have already been generated. We instead return the first result to the model so it can construct the second call on the next turn, or construct that call explicitly in application code.
:::


### Acting on feedback

An agent loop connects a model's proposed action to an observation from the environment:

```{mermaid}
flowchart TD
    A["Current goal and message history"] --> B["Model chooses next step"]
    B --> C{"Use a tool?"}
    C -->|Yes| D["Runtime validates and executes the call"]
    D --> E["Add result or error trace to the history"]
    E --> B
    C -->|No| F["Return a final answer"]
```

The goal usually remains fixed while the message history grows with each observation.

For example, suppose a support agent receives a report that an order has not arrived. It might look up the order, inspect the tracking result, and then choose between explaining a delay, asking for missing information, or escalating the case. A failed lookup changes what it should do next. The model needs to perform a tool call to make that decision.

The loop also needs **stopping rules** enforced by the application. It can stop when the task is complete, request human help when blocked, or halt after a limit on steps, time, or spending. A model that keeps requesting actions should not be able to extend its own budget.


:::{.callout-note title="Giving agents eyes: Tools as sensors"}
Suppose we ask a coding agent to center the content in a web app. It edits `main.py`, runs the app, and requests a screenshot. The application uses Selenium to capture the browser preview and passes the image back to the model. A successful run tells us the app started. **The screenshot tool gives the agent eyes.** The model can inspect the rendered layout, decide whether the content looks centered, and request another screenshot after a change. See [Selenium’s screenshot guide](https://www.selenium.dev/documentation/guides/screenshots/).

A tool can serve as a **sensor** wherever software can collect a signal and return it in a form the model can interpret. A microphone paired with transcription lets the agent follow speech, and a temperature sensor lets it monitor a room. With a separate tool to switch on the air conditioner, the agent could act on a temperature reading, then take another to see whether the room cooled.

![A coding agent checks its layout changes in the app preview. **Source:** [*Flet agent got eyes and ears*][flet-preview-source] by Feodor Fitsner.](../images/post/week-1-introduction-to-ai-agents-52bb95e2946441f2970067f256314f8d.png){#fig-week-1-introduction-to-ai-agents-52bb95e2946441f2970067f256314f8d fig-alt="The user asks the agent to center the content. The agent edits main.py and runs the app. The preview returns console output and a screenshot. The agent repeats the loop until the content looks centered, then reports back to the user."}

[flet-preview-source]: https://flet.dev/blog/flet-agent-got-eyes-and-ears/
:::

## Orchestration patterns

For many applications, optimizing single LLM calls with retrieval and in-context examples is usually enough. But it can be good to have agents arranged in certain patterns depending on the use case. Further, for an [agent harness](https://en.wikipedia.org/wiki/Agent_harness) its good to have certain preset workflow patterns available to guide the orchestrator model. The examples below apply them to a writing assistant. The patterns below are based on [Anthropic's *Building effective agents*](https://www.anthropic.com/research/building-effective-agents). 

| Pattern | How the work is organized | Example |
|:--|:---|:---|
| **Prompt chaining** | Fixed stages pass their results to the next stage, with checks/gates between them. | Create an outline, check its coverage, then draft the explanation. |
| **Routing** | A classifier selects a specialized path. | Route a request to a beginner explanation or a mathematical treatment. |
| **Parallelization** | Independent calls run concurrently and their results are combined. | Review a draft for clarity and factual support in separate calls. |
| **Orchestrator-workers** | A coordinating model determines subtasks and combines worker results. | Break a research question into smaller questions, investigate them, then synthesize an answer. |
| **Evaluator-optimizer** | A candidate draft is evaluated and revised using feedback. | Revise an explanation until it meets a stated rubric or reaches a retry limit. |

These patterns can be combined. The orchestrator-workers pattern already allows dynamic task decomposition inside a defined structure, so the workflow–agent distinction is a useful description of control rather than a rigid classification of every application.


## Observability and evaluation

Agents' dynamic execution path makes failures harder to diagnose. 

- **Observability.** Records enough of a run to reconstruct what happened: available messages, tool names and arguments, results, errors, latency, and usage. A trace can show whether a poor answer followed a failed lookup, missing context, or an incorrect interpretation of a valid result.

- **Evaluation.** Asks whether the system accomplished the task. In Anthropic's [*Demystifying evals for AI agents*](https://www.anthropic.com/research/demystifying-evals-for-ai-agents), the execution trace is distinguished from the final outcome. For our support example, a convincing message saying the issue has been resolved is only one artifact. We would also check the order records, the action actually taken, and whether the customer received an appropriate resolution.

Different checks answer different questions:

| Check | Purpose |
|:--|:--|
| **Code-based checks** | Verify conditions such as a word limit, required fields, or a change in a test database. |
| **Model-based checks** (*LLM-as-a-judge*) | Assess a response against a rubric, such as whether its explanation is supported by the supplied evidence. |
| **Human review** | Handles ambiguous cases and helps calibrate the automated judgments. |

A judge's score is an estimate of quality and needs comparison with human judgments; it does not establish that an external action succeeded. For a support system, a useful experiment would compare a simple baseline with the proposed agent on the same representative requests. Measure correct resolutions, unsupported claims, appropriate escalations, time, and cost. Repeat runs to expose variation. 

In production, we want to track whether the system reduces resolution time and repeat contacts while maintaining an acceptable error rate. Fluent answers alone would not demonstrate that improvement.


## Guardrails and execution limits

Giving a model more choices also creates more possible failure paths. It can select the wrong tool, accept an unsupported claim, repeat an unsuccessful action, or spend more tokens than the task warrants[^sad]. **Guardrails** are checks and restrictions around execution that keep the system within its intended boundaries.

[^sad]: Don't ask me how I know. (╥﹏╥)ノ💵🔥

For the support example, those controls could include:

| Control | Purpose |
|:--|:--|
| **Tool permissions** | Allow access only to the current customer's records, and validate arguments before an operation runs. |
| **Execution budgets** | Cap steps, retries, elapsed time, and spending; provide a clear path for escalation. |
| **Output checks** | Verify required fields and factual support before returning an answer. A valid JSON structure checks the shape of an answer, while its claims still need evidence. |
| **Approval boundaries** | Require an authorized person to approve actions such as issuing a refund or changing an account. |

These controls operate at different points: before a tool call, during the loop, or before an answer is released. An instruction in a prompt can *guide behavior*; but permission checks and limits in application code *enforce* the allowed actions.

:::{.callout-note title="Example: a ticket-resolution reply under 500 words"}
Suppose a support application provides the model with the ticket and a record of the resolution. The prompt could read:

> Write a customer-facing reply in fewer than 500 words. Summarize the issue, the resolution, and any remaining steps for the customer. If a resolution-document URL is supplied, include it for further detail.

The application checks the complete reply, including any link, with `len(reply.split()) < 500`, using the same whitespace-based word count as our earlier example. A 500-word reply fails this check and is returned for revision before release. Each revision is checked again within the retry budget. The linked document can hold longer instructions while the ticket reply remains brief.
:::

:::{.callout-tip title="Guardrails also need evaluation"}
A system that blocks valid requests too often can fail its purpose, while one that misses disallowed actions can exceed its authority. The course develops these controls further in later weeks.
:::


## Common design traps

Before proceeding, we want to highlight two traps when designing an AI agents-based solution:

| Trap | Explanation | Example fix |
|:--|:--|:--|
| **Choosing an architecture before defining the problem** | A request for a "delivery support agent" names a proposed solution, but does not yet identify what needs to improve.<br><br>Suppose customers wait too long for answers about delayed orders. The initial question is which part of resolving those questions causes the delay, and whether an LLM application would help. | If the application can supply all relevant order information at the outset, one LLM call gives us a baseline.<br><br>An agent loop becomes worth testing when the next lookup depends on what an earlier lookup reveals. Comparing the two designs on the same cases shows whether the added flexibility improves resolution enough to justify its cost and latency. |
| **Mistaking named roles for independent expertise** | Calling agents "researcher," "writer," and "manager" can help distinguish their responsibilities. It does not, by itself, give them the capabilities or independent judgment of a human team.<br><br>Several agents may repeat the same error; agreement among their answers is not necessarily corroboration. | Suppose we add a "reviewer" after the "writer." If the reviewer merely endorses the draft, it may repeat the writer's error.<br><br>If, instead, it compares the draft's claims against trusted records, it has a distinct task: detecting unsupported claims. We can then test whether this check catches errors the writer missed, rather than treating the reviewer's title or agreement as evidence of correctness. Claims that the records cannot establish can remain unresolved or be referred for human review. |
